# mPES - Colab Launcher desde GitHub
<!-- pylint: disable=undefined-variable -->
<!-- pyright: reportUndefinedVariable=false -->

Este notebook clona una copia ligera del repositorio en el almacenamiento local
de Colab y ejecuta la optimizacion desde `h1/`. Los resultados se conservan
en Google Drive. Cada celda importa lo que usa porque los linters analizan
las celdas de forma aislada.

Configura en la primera celda el repositorio y la rama. El clonado usa
`--depth 1 --single-branch` para reducir tiempo, espacio y trafico de red.
Para `ens_sprb` o `ens_accq`, los modelos pre-entrenados viajan dentro del
propio clon (`h1/ml/pes_{dqn,rdqn,trf}/inputs/*_model.keras`); si en Drive hay
copias mas recientes (`MyDrive/mPES/<pes_dqn|pes_rdqn|pes_trf>/`, donde las
guarda `retrain_gpu.ipynb`), esas tienen prioridad y se copian al clon. Los
modelos nunca se reentrenan desde este notebook. La celda 3 verifica ademas
que cada `.keras` carga e infiere con la version de TensorFlow de Colab antes
de lanzar nada (un modelo corrupto o incompatible provoca un segfault en el
optimizador).

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab. La ultima celda dibuja el progreso del
estudio Optuna (historial y mejores valores) al terminar la optimizacion.

In [1]:
"""Colab launcher for mPES Bayesian optimisation."""
# Mount Google Drive before cloning the repository.
# ==========================================================================
# MOUNT GOOGLE DRIVE
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Configure a Colab optimisation from a shallow Git clone.
# pylint: disable=undefined-variable
# pyright: reportUndefinedVariable=false
import os

REPOSITORY   = 'https://github.com/Maximiliano0/mPES_2026.git'

[INFO] [Celda 1] Configuración guardada: PKG='ens_sprb', N_TRIALS=50, BRANCH='new_uq'


In [ ]:
# Shallow clone: only the selected branch and its current snapshot.
# pylint: disable=undefined-variable
# pyright: reportUndefinedVariable=false
import os
import subprocess

if os.path.isdir(WORKSPACE):

[INFO] [Celda 2] Borrando workspace anterior en /content/mPES...
[INFO] [Celda 2] Clonando el repositorio rama 'new_uq'...
[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...

  mPES  Colab Pro+ bootstrap

 Drive workspace: /content/drive/MyDrive/mPES/runs

  Installing Python dependencies

  Checking pinned optimisation and training dependencies
  Required runtime packages already match project versions.

  Exporting mPES environment variables

 Env vars sourced from /content/mpes_env.sh

  Bootstrap complete. Next: run utils/colab/run_colab.sh <PKG> <TRIALS>


[INFO] [Celda 2] Shallow clone y setup completados: https://github.com/Maximiliano0/mPES_2026.git@new_uq


In [ ]:
# pylint: disable=undefined-variable
# pyright: reportUndefinedVariable=false
import glob
import os
import shutil
import subprocess

DRIVE_ROOT = '/content/drive/MyDrive/mPES'

[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_dqn/inputs/dqn_model.keras
[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_rdqn/inputs/rdqn_model.keras
[INFO] [Celda 3] Modelo del clon (baseline): /content/mPES/h1/ml/pes_trf/inputs/trf_model.keras
[INFO] [Celda 3] 3 modelos listos para ens_sprb.


In [5]:
# Launch the Bayesian optimisation via run_colab.sh (blocks, tails Drive log).
import os
import subprocess
import time

run_environment = os.environ.copy()
run_environment.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'WORKSPACE': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
_script = '''
set -uo pipefail
cd "$WORKSPACE"
source /content/mpes_env.sh
bash "$H_DIR/general/colab/run_colab.sh" "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''
print("[INFO] [Celda 4] Iniciando run_colab.sh... Observa los registros a continuación:")
start = time.time()
run = subprocess.run(
    _script,
    shell=True,
    executable='/bin/bash',
    check=False,
    env=run_environment,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(run.stdout)
elapsed = time.time() - start

# If the run ended suspiciously fast, surface the real error from Drive logs.
if run.returncode != 0 or elapsed < 120:
    run_date = RESUME_DATE or time.strftime('%Y-%m-%d')
    run_dir = os.path.join(OUTPUT_ROOT, f'pes_{PKG}' if not PKG.startswith('pes_') else PKG,
                           f'{run_date}_BAYESIAN_OPT')
    for log_name in ('bayesian_opt_err.log', 'supervisor.log', 'bayesian_opt.log'):
        log_path = os.path.join(run_dir, log_name)
        if os.path.isfile(log_path):
            with open(log_path, 'r', encoding='utf-8', errors='replace') as handle:
                tail = handle.read()[-6000:]
            print(f'\n[WARN] [Celda 4] Últimos registros de {log_name} (terminó en {elapsed:.0f}s):\n{tail}')

if run.returncode != 0:
    print(f"[ERROR] [Celda 4] run_colab.sh falló con código {run.returncode}")
    print(f"[ERROR] Revisa: {OUTPUT_ROOT}/pes_ens_sprb/<FECHA>_BAYESIAN_OPT/")
    raise RuntimeError(f'run_colab.sh exited with code {run.returncode}')
print("[INFO] [Celda 4] Ejecución de run_colab.sh completada con éxito!")

[INFO] [Celda 4] Iniciando run_colab.sh... Observa los registros a continuación:

  Launching Bayesian optimisation on Colab Pro+

  Package         : pes_ens_sprb
  Module          : ens.pes_ens_sprb.ext.optimize_ens
  Trials          : 50
  Run date        : 2026-09-03
  Output dir      : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT
  Storage         : sqlite:////content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/optuna_study_2026-09-03.db
  Git             : new_uq@293de4f
  Python          : 3.13.15
  GPU mode        : 0

 Optimisation PID    : 18600
 stdout              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt.log
 stderr              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt_err.log
 metadata            : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/run_meta.json


  Optimisation supervisor launched.
   Cell BLOCKS in foregroun

In [ ]:
# Visualise the Optuna study: optimisation history and per-trial values.
# pylint: disable=undefined-variable
# pyright: reportUndefinedVariable=false
import glob
import os

import matplotlib.pyplot as plt
import optuna

study_dbs = sorted(
    glob.glob(os.path.join(OUTPUT_ROOT, 'pes_*', '*_BAYESIAN_OPT', 'optuna_study_*.db')),
    key=os.path.getmtime, reverse=True,
)
if not study_dbs:
    print(f'[INFO] [Celda 5] No hay estudios en {OUTPUT_ROOT} todavía.')
else:
    db_path = study_dbs[0]
    study_name = os.path.basename(os.path.dirname(os.path.dirname(db_path)))
    study = optuna.load_study(study_name=study_name, storage=f'sqlite:///{db_path}')
    completed = [t for t in study.trials if t.value is not None]
    print(f'[INFO] [Celda 5] Estudio: {study_name} ({os.path.basename(db_path)})')
    print(f'[INFO] [Celda 5] Trials completados: {len(completed)} / {len(study.trials)}')
    if completed:
        values = [t.value for t in completed]
        best_so_far = []
        current = float('-inf')
        for value in values:
            current = max(current, value)
            best_so_far.append(current)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
        ax1.plot([t.number for t in completed], values, 'o-', markersize=3, linewidth=0.8)
        ax1.set_xlabel('Trial'); ax1.set_ylabel('Valor'); ax1.set_title('Valor por trial')
        ax1.grid(alpha=0.3)
        ax2.plot([t.number for t in completed], best_so_far, '-', color='tab:orange')
        ax2.set_xlabel('Trial'); ax2.set_ylabel('Mejor valor'); ax2.set_title('Optimisation history')
        ax2.grid(alpha=0.3)
        fig.suptitle(f'{study_name} — mejor: {max(values):.4f} (trial '
                     f'{completed[values.index(max(values))].number})')
        fig.tight_layout()
        plt.show()
        print(f'[INFO] [Celda 5] Mejor valor: {max(values):.6f}')
        print(f'[INFO] [Celda 5] Mejores parámetros: {study.best_params}')